# US–India Goods Trade, 1995–2024: what is traded and how it has shifted

**Author:** Arun Kumar Prabhakaran
**Date:** 2026-11-01

**Question:** what do the US and India sell to each other, how large has that trade become, and how has the mix of commodities changed since 1995?

**Source:** the UN Comtrade public API (no key required), which publishes goods trade by Harmonized System (HS) commodity code as reported by each country's customs. Both countries report, which lets us compare the two sides of the same trade ("mirror data").

**Pipeline**
1. **Collect:** one API call per country, direction and year for the 97 HS chapters (cached on disk), plus 4-digit product detail for the biggest chapters in benchmark years.
2. **Clean and validate:** attach commodity names, check the chapter data against Comtrade's own totals, and look for gaps.
3. **Analyze:** totals and balance, composition and shifts, concentration, the gap between the two countries' reports, and what drove growth.
4. **Visualize** and **conclude**, with caveats.

**Scope and caveats up front**
* **Goods only.** Services (software, IT, travel, education) are a large part of US–India economic ties and are not in this data.
* **Current US dollars**, not adjusted for inflation, so long-run growth is partly price change.
* **Primary series are the US customs figures** (US exports to India, US imports from India). India's own reports are used as the mirror for comparison.

## 1. Setup

In [ ]:
# Author: Arun Kumar Prabhakaran
# Date: 2026-11-01
import json
import re
import time
import warnings
from pathlib import Path

# macOS system Python links LibreSSL, which makes urllib3 v2 print a harmless warning on import; silence it
# (must run before `requests` is imported)
warnings.filterwarnings("ignore", message="urllib3 v2 only supports OpenSSL")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from matplotlib.lines import Line2D

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

ROOT = Path.cwd()
RAW_DIR = ROOT / "data" / "raw" / "comtrade"
CLEAN_DIR = ROOT / "data" / "clean"
FIG_DIR = ROOT / "figures"
for d in (RAW_DIR, CLEAN_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

API = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
REFERENCE_URL = "https://comtradeapi.un.org/files/v1/app/reference/HS.json"
HEADERS = {"User-Agent": "us-india-trade-analysis/0.1 (personal educational project; python-requests)"}

USA, IND = 842, 699                 # Comtrade country codes
YEARS = list(range(1995, 2025))     # chapter-level history
BENCHMARK_YEARS = [1995, 2005, 2015, 2024]   # years for the 4-digit product drill-down
N_DRILL_CHAPTERS = 10               # biggest chapters per direction to drill into
HS4_BATCH = 150                     # product codes per API call (the preview caps a response at 500 rows; split automatically if refused)
CHECK_YEARS = [1995, 2024]          # years for the chapter-sum vs official-total check
MAX_QUOTA_WAIT = 3600               # longest wait (seconds) for the API call quota to reset before giving up
DELAY = 1.5                         # seconds between API calls
REFRESH = False  # True: ignore the cache in data/raw and re-download every API response once (set back to False afterwards)

## 2. Collect

The public preview endpoint allows **one year per request**, **at most 500 rows per response** and is rate-limited and has a **call-volume quota** (when it is used up the API answers 403 and says when it resets), so the code below caches every response in `data/raw/comtrade/`, waits between calls, retries politely on HTTP 429 and, if the quota runs out, waits for it to reset and carries on (progress is never lost). A response with no rows is cached too, so a missing year is not requested again. To pull fresh data, set `REFRESH = True` in the setup cell and run all cells (it re-downloads each response once; set it back to `False` afterwards). The first run makes about 150 calls and takes several minutes (longer if the quota has to reset); later runs read the cache and take seconds.

In [ ]:
stats = {"downloaded": 0, "cached": 0}
_refreshed = set()  # responses already re-downloaded in this run, so REFRESH fetches each only once


def cached_json(path, url, params=None):
    if path.exists() and not (REFRESH and path not in _refreshed):
        stats["cached"] += 1
        return json.loads(path.read_text(encoding="utf-8"))
    for attempt in range(12):
        resp = requests.get(url, params=params, headers=HEADERS, timeout=60)
        if resp.status_code == 429:  # "Rate limit is exceeded. Try again in N seconds."
            m = re.search(r"(\d+)\s*second", resp.text)
            wait = (int(m.group(1)) if m else 5) + 1 + attempt
            print(f"rate limited, waiting {wait}s")
            time.sleep(wait)
            continue
        if resp.status_code == 403 and "quota" in resp.text.lower():  # "Out of call volume quota. Quota will be replenished in HH:MM:SS."
            m = re.search(r"(\d+):(\d+):(\d+)", resp.text)
            wait = int(m.group(1)) * 3600 + int(m.group(2)) * 60 + int(m.group(3)) if m else int(resp.headers.get("Retry-After", 1200))
            if wait > MAX_QUOTA_WAIT:
                raise RuntimeError(f"API quota used up and it resets in {wait / 60:.0f} min; run the notebook again later (progress is cached)")
            print(f"API call quota used up, waiting {wait / 60:.0f} min for it to reset (progress is cached)")
            time.sleep(wait + 5)
            continue
        resp.raise_for_status()
        break
    else:
        raise RuntimeError(f"still rate limited after retries: {url} {params}")
    data = resp.json()
    path.write_text(json.dumps(data), encoding="utf-8")  # only reached after a successful download
    _refreshed.add(path)
    stats["downloaded"] += 1
    time.sleep(DELAY)
    return data


def comtrade(reporter, partner, flow, year, cmd):
    # cmd: "AG2" (all chapters), "TOTAL", or a list of HS codes
    key = cmd if isinstance(cmd, str) else f"hs4-{cmd[0]}-{cmd[-1]}-{len(cmd)}"
    params = {"reporterCode": reporter, "partnerCode": partner, "period": year, "flowCode": flow,
              "cmdCode": cmd if isinstance(cmd, str) else ",".join(cmd)}
    data = cached_json(RAW_DIR / f"{reporter}-{partner}-{flow}-{year}-{key}.json", API, params)
    rows = data.get("data") or []
    if len(rows) >= 500:
        print(f"WARNING: {reporter}-{partner}-{flow}-{year}-{key} hit the 500-row cap, results are truncated")
    return rows


def comtrade_codes(reporter, partner, flow, year, codes):
    # a batch of HS codes; if the API refuses a big batch (HTTP 400), split it in half and try again
    try:
        return comtrade(reporter, partner, flow, year, codes)
    except requests.HTTPError as e:
        if e.response is not None and e.response.status_code == 400 and len(codes) > 1:
            mid = len(codes) // 2
            return comtrade_codes(reporter, partner, flow, year, codes[:mid]) + comtrade_codes(reporter, partner, flow, year, codes[mid:])
        raise


def report(label):
    print(f"{label}: {stats['downloaded']} downloaded, {stats['cached']} from cache")
    stats.update(downloaded=0, cached=0)


# reporter / direction of each series
SERIES = {
    "us_exports": dict(reporter=USA, partner=IND, flow="X", label="US exports to India (US-reported)"),
    "us_imports": dict(reporter=USA, partner=IND, flow="M", label="US imports from India (US-reported)"),
    "in_imports": dict(reporter=IND, partner=USA, flow="M", label="India imports from US (India-reported)"),
    "in_exports": dict(reporter=IND, partner=USA, flow="X", label="India exports to US (India-reported)"),
}
FIELDS = ["period", "flowCode", "cmdCode", "aggrLevel", "primaryValue", "netWgt", "classificationCode"]

In [ ]:
frames = []
for key, s in SERIES.items():
    for year in YEARS:
        rows = comtrade(s["reporter"], s["partner"], s["flow"], year, "AG2")
        if rows:
            part = pd.DataFrame(rows).reindex(columns=FIELDS)
            part["series"] = key
            frames.append(part)
report("chapter-level collection")

raw = pd.concat([f.dropna(axis=1, how="all") for f in frames], ignore_index=True).reindex(columns=FIELDS + ["series"])
raw["cmdCode"] = raw["cmdCode"].astype(str)
chap = raw[raw["cmdCode"].str.fullmatch(r"\d{2}")].copy()
chap["year"] = chap["period"].astype(int)
chap["value_usd_m"] = chap["primaryValue"].astype(float) / 1e6
chap = (chap.groupby(["series", "year", "cmdCode"], as_index=False)
        .agg(value_usd_m=("value_usd_m", "sum"), net_wgt_kg=("netWgt", "sum"), hs_revision=("classificationCode", "first")))
chap = chap[chap["value_usd_m"] > 0]

coverage = chap.groupby("series").agg(first_year=("year", "min"), last_year=("year", "max"), years=("year", "nunique"),
                                      chapters_latest=("cmdCode", lambda s: s[chap.loc[s.index, "year"] == chap.loc[s.index, "year"].max()].nunique()))
coverage

## 3. Commodity names

Comtrade's public preview returns codes without descriptions, so the names come from Comtrade's public HS reference file. The Harmonized System is revised every five years or so (1996, 2002, 2007, 2012, 2017, 2022), so codes can be split, merged or moved; chapter-level (2-digit) figures are much more stable across revisions than 4-digit products.

In [ ]:
ref_json = cached_json(RAW_DIR / "HS-reference.json", REFERENCE_URL)
ref = pd.DataFrame(ref_json["results"] if isinstance(ref_json, dict) else ref_json)  # the file wraps its list in a "results" key
ref["id"] = ref["id"].astype(str)
ref["name"] = ref["text"].str.replace(r"^\d+ - ", "", regex=True)
chapter_names = ref[ref["id"].str.fullmatch(r"\d{2}")].set_index("id")["name"]
hs4 = ref[ref["id"].str.fullmatch(r"\d{4}")].copy()
hs4["chapter"] = hs4["id"].str[:2]
hs4_names = hs4.set_index("id")["name"]
report("reference file")

# short labels for charts: hand-written for the chapters that matter, otherwise the first clause of the official name
SHORT = {
    "27": "Mineral fuels & oil", "71": "Gems, jewelry & precious metals", "84": "Machinery & mechanical appliances",
    "85": "Electrical machinery & electronics", "30": "Pharmaceuticals", "29": "Organic chemicals", "88": "Aircraft & parts",
    "90": "Optical & medical instruments", "87": "Vehicles & parts", "08": "Edible fruit & nuts", "12": "Oil seeds & grains",
    "10": "Cereals", "52": "Cotton", "61": "Apparel, knitted", "62": "Apparel, not knitted", "63": "Other textile articles",
    "39": "Plastics", "38": "Misc. chemical products", "72": "Iron & steel", "73": "Iron & steel articles", "76": "Aluminium",
    "03": "Fish & seafood", "09": "Coffee, tea & spices", "17": "Sugars", "24": "Tobacco", "28": "Inorganic chemicals",
    "32": "Dyes & paints", "33": "Essential oils & cosmetics", "40": "Rubber", "41": "Hides & leather", "42": "Leather articles",
    "44": "Wood", "47": "Wood pulp", "48": "Paper", "55": "Man-made staple fibres", "57": "Carpets", "64": "Footwear",
    "68": "Stone & cement articles", "69": "Ceramics", "70": "Glass", "74": "Copper", "75": "Nickel", "78": "Lead", "79": "Zinc",
    "82": "Tools & cutlery", "83": "Misc. metal articles", "89": "Ships & boats", "94": "Furniture & lighting", "95": "Toys & games",
    "96": "Misc. manufactured articles", "97": "Works of art", "99": "Unclassified", "13": "Lac, gums & resins", "15": "Fats & oils",
    "23": "Food industry residues", "20": "Preserved vegetables & fruit", "21": "Misc. edible preparations", "26": "Ores & slag",
    "31": "Fertilizers", "34": "Soaps & waxes", "35": "Albumins & glues", "36": "Explosives", "37": "Photographic goods",
    "49": "Printed matter", "50": "Silk", "51": "Wool", "53": "Other vegetable fibres", "54": "Man-made filaments", "56": "Wadding & felt",
    "58": "Special woven fabrics", "59": "Coated textile fabrics", "60": "Knitted fabrics", "65": "Headgear", "67": "Feathers & hair articles",
    "81": "Other base metals", "86": "Railway equipment", "91": "Clocks & watches", "92": "Musical instruments", "93": "Arms & ammunition",
}


def short_name(code):
    if code in SHORT:
        return SHORT[code]
    full = chapter_names.get(code, code)
    return re.split(r"[;,]", full)[0].strip()[:36]


SHORT_PRODUCT = {
    "2709": "Crude petroleum", "2701": "Coal", "2711": "Petroleum gases (LNG, LPG)", "2710": "Refined petroleum oils", "2713": "Petroleum coke & bitumen",
    "7102": "Diamonds", "7108": "Gold, unwrought", "7113": "Jewellery", "7104": "Synthetic & reconstructed stones", "0802": "Nuts (excl. coconut, Brazil, cashew)",
    "8471": "Computers & data processing machines", "9018": "Medical & surgical instruments", "3004": "Medicaments (packaged pharmaceuticals)",
    "8517": "Telephone sets incl. smartphones", "6302": "Bed, table & kitchen linen", "8541": "Semiconductor devices", "8483": "Transmission shafts & gearing",
    "8504": "Transformers & converters", "8802": "Aircraft & spacecraft", "8803": "Aircraft parts", "8703": "Passenger cars", "2933": "Heterocyclic compounds",
    "3002": "Vaccines, blood & immunological products", "8708": "Motor-vehicle parts", "8542": "Integrated circuits", "7106": "Silver, unwrought",
}


def product_label(code):
    if code in SHORT_PRODUCT:
        return SHORT_PRODUCT[code]
    return re.split(r"[;,(]", hs4_names.get(code, code))[0].strip()[:44]


print(len(chapter_names), "HS chapters,", len(hs4), "four-digit headings in the reference file")

## 4. Validate

Two checks: (1) the sum of the chapter rows should equal Comtrade's own `TOTAL` row for the same query, and (2) every series should have data for every year (a gap would distort trends).

In [ ]:
chapter_sum = chap.groupby(["series", "year"]).value_usd_m.sum()
rows = []
for key, s in SERIES.items():
    for year in CHECK_YEARS:
        total = comtrade(s["reporter"], s["partner"], s["flow"], year, "TOTAL")
        official = sum((r["primaryValue"] or 0) for r in total) / 1e6
        mine = chapter_sum.get((key, year), np.nan)
        rows.append({"series": key, "year": year, "chapters_sum_usd_m": mine, "comtrade_total_usd_m": official,
                     "ratio": mine / official if official else np.nan})
report("TOTAL cross-check")
check = pd.DataFrame(rows)
display(check)
print(f"chapter sum / official total: min {check['ratio'].min():.4f}, max {check['ratio'].max():.4f}")

years_present = chap.groupby("series").year.apply(set)
for key in SERIES:
    gaps = sorted(set(YEARS) - years_present.get(key, set()))
    print(f"{key}: missing years -> {gaps if gaps else 'none'}")

## 5. Analysis

### 5.1 Size of the trade and the balance

In [ ]:
LATEST = int(chap.loc[chap.series == "us_exports", "year"].max())
tot = chap.pivot_table(index="year", columns="series", values="value_usd_m", aggfunc="sum") / 1000   # USD bn
tot["us_balance"] = tot["us_exports"] - tot["us_imports"]       # negative = the US buys more than it sells
tot["total_trade"] = tot["us_exports"] + tot["us_imports"]
tot["exports_to_imports"] = tot["us_exports"] / tot["us_imports"]


def cagr(a, b, years):
    return (b / a) ** (1 / years) - 1


first = tot.index.min()
summary = pd.DataFrame({
    first: tot.loc[first, ["us_exports", "us_imports", "us_balance", "total_trade"]],
    LATEST: tot.loc[LATEST, ["us_exports", "us_imports", "us_balance", "total_trade"]],
})
summary["growth_x"] = summary[LATEST] / summary[first]
summary["cagr_pct"] = [cagr(tot.loc[first, k], tot.loc[LATEST, k], LATEST - first) * 100 if tot.loc[first, k] > 0 and tot.loc[LATEST, k] > 0 else np.nan
                       for k in summary.index]
print(f"USD billion, current prices, US-reported (first year with data: {first}, latest: {LATEST})")
summary

### 5.2 What is traded: the top chapters and how their shares changed

In [ ]:
def share_table(series, years=BENCHMARK_YEARS + [LATEST]):
    p = chap[chap.series == series].pivot(index="cmdCode", columns="year", values="value_usd_m").fillna(0)
    share = p / p.sum() * 100
    return p, share


def top_chapters(series, n=10, years=sorted(set(BENCHMARK_YEARS + [LATEST]))):
    p, share = share_table(series)
    out = share.loc[share[LATEST].nlargest(n).index, years].copy()
    out.insert(0, "chapter", [short_name(c) for c in out.index])
    out[f"{LATEST} value (USD bn)"] = p.loc[out.index, LATEST] / 1000
    return out


print("US EXPORTS to India: share of the year's total (%)")
display(top_chapters("us_exports"))
print("US IMPORTS from India: share of the year's total (%)")
display(top_chapters("us_imports"))

### 5.3 Biggest gainers and losers in share

In [ ]:
def shifts(series, start=min(BENCHMARK_YEARS), end=LATEST, n=6):
    _, share = share_table(series)
    d = (share[end] - share[start]).rename("change_pp")
    d = pd.DataFrame({"chapter": [short_name(c) for c in d.index], f"share {start}": share.loc[d.index, start],
                      f"share {end}": share.loc[d.index, end], "change_pp": d}, index=d.index)
    return d.sort_values("change_pp", ascending=False).pipe(lambda x: pd.concat([x.head(n), x.tail(n)]))


print("US exports to India, change in share of total (percentage points)")
display(shifts("us_exports"))
print("US imports from India, change in share of total (percentage points)")
display(shifts("us_imports"))

### 5.4 Concentration

How much of the trade sits in a handful of chapters: the share of the top 5 chapters and the Herfindahl-Hirschman Index (HHI, the sum of squared shares; 10,000 = a single product, below about 1,500 is considered unconcentrated).

In [ ]:
def concentration(series):
    p, share = share_table(series)
    return pd.DataFrame({"top5_share_pct": share.apply(lambda c: c.nlargest(5).sum()),
                         "hhi": (share ** 2).sum(),
                         "n_chapters": (p > 0).sum()})


conc = {k: concentration(k) for k in ("us_exports", "us_imports")}
pd.concat({k: v.loc[sorted(set(BENCHMARK_YEARS + [LATEST]))] for k, v in conc.items()}, axis=1)

### 5.5 Do the two countries' reports agree?

The same shipment is reported twice: by the exporter's customs on the way out (valued FOB, free on board) and by the importer's customs on the way in (valued CIF, which adds freight and insurance). So the importer's figure should be somewhat higher, typically by 5% to 10%. A bigger gap points to timing, valuation, re-exports or classification differences.

In [ ]:
mirror = pd.DataFrame({
    "us_exports_vs_india_imports": tot["in_imports"] / tot["us_exports"],     # importer-reported / exporter-reported
    "india_exports_vs_us_imports": tot["us_imports"] / tot["in_exports"],
})
mirror.index.name = "year"
MIRROR_YEAR = int(mirror.dropna().index.max())
print(f"importer-reported / exporter-reported; 1.00 = perfect agreement; latest year with both sides: {MIRROR_YEAR}")
display(mirror.loc[[y for y in sorted(set(BENCHMARK_YEARS + [MIRROR_YEAR])) if y in mirror.index]])


def chapter_gap(exporter_series, importer_series, year=MIRROR_YEAR, n=10):
    a = chap[(chap.series == exporter_series) & (chap.year == year)].set_index("cmdCode")["value_usd_m"]
    b = chap[(chap.series == importer_series) & (chap.year == year)].set_index("cmdCode")["value_usd_m"]
    g = pd.DataFrame({"exporter_reported": a, "importer_reported": b}).fillna(0)
    g["gap_usd_m"] = g["importer_reported"] - g["exporter_reported"]
    g["chapter"] = [short_name(c) for c in g.index]
    return g.reindex(g["gap_usd_m"].abs().sort_values(ascending=False).index).head(n)


print(f"{MIRROR_YEAR}: US exports to India, chapters with the largest gap (USD million)")
display(chapter_gap("us_exports", "in_imports"))
print(f"{MIRROR_YEAR}: India exports to US, chapters with the largest gap (USD million)")
display(chapter_gap("in_exports", "us_imports"))

### 5.6 What drove the growth?

Contribution of each chapter to the change in the total between two years.

In [ ]:
GROWTH_START = 2005


def contributions(series, start=GROWTH_START, end=LATEST, n=8):
    p, _ = share_table(series)
    d = (p[end] - p[start]) / 1000
    out = pd.DataFrame({"chapter": [short_name(c) for c in d.index], "change_usd_bn": d})
    out["share_of_total_change_pct"] = out["change_usd_bn"] / out["change_usd_bn"].sum() * 100
    return out.sort_values("change_usd_bn", ascending=False).head(n), out["change_usd_bn"].sum()


for s in ("us_exports", "us_imports"):
    tbl, total_change = contributions(s)
    print(f"{s}: total change {GROWTH_START} to {LATEST} = {total_change:,.1f} USD bn")
    display(tbl)

### 5.7 Product detail (4-digit HS) for the biggest chapters

Within the ten largest chapters in each direction, the 4-digit headings are requested explicitly (the preview caps a response at 500 rows, so asking for "all products" would be silently truncated). Product detail is fetched for the benchmark years only. Because HS codes change between revisions, a code that exists today may not have existed in 1995, so for every year the table below reports **coverage**: the share of the chapters' known total that the 4-digit rows account for. Where coverage is below 100%, the product ranking for that year leaves out the missing value; the cell below lists any chapter that falls short by more than 5% in the latest year.

In [ ]:
drill_series = {"us_exports": "X", "us_imports": "M"}
hs4_frames = []
drill_chapters = {}
for key in drill_series:
    s = SERIES[key]
    drill_chapters[key] = chap[chap.series == key].groupby("cmdCode").value_usd_m.sum().nlargest(N_DRILL_CHAPTERS).index.tolist()
    codes = sorted(hs4[hs4["chapter"].isin(drill_chapters[key])]["id"])
    batches = [codes[i:i + HS4_BATCH] for i in range(0, len(codes), HS4_BATCH)]
    for year in BENCHMARK_YEARS:
        for batch in batches:
            rows = comtrade_codes(s["reporter"], s["partner"], s["flow"], year, batch)
            if rows:
                part = pd.DataFrame(rows).reindex(columns=FIELDS)
                part["series"], part["year"] = key, year
                hs4_frames.append(part)
report("4-digit drill-down")

hs4_data = pd.concat([f.dropna(axis=1, how="all") for f in hs4_frames], ignore_index=True).reindex(columns=FIELDS + ["series", "year"])
hs4_data["cmdCode"] = hs4_data["cmdCode"].astype(str)
hs4_data = hs4_data[hs4_data["cmdCode"].str.fullmatch(r"\d{4}")].copy()
hs4_data["value_usd_m"] = hs4_data["primaryValue"].astype(float) / 1e6
hs4_data = hs4_data.groupby(["series", "year", "cmdCode"], as_index=False).agg(value_usd_m=("value_usd_m", "sum"))
hs4_data["product"] = hs4_data["cmdCode"].map(hs4_names)
hs4_data["chapter"] = hs4_data["cmdCode"].str[:2]

known = chap[chap.apply(lambda r: r["cmdCode"] in drill_chapters.get(r["series"], []), axis=1)].groupby(["series", "year"]).value_usd_m.sum()
found = hs4_data.groupby(["series", "year"]).value_usd_m.sum()
cov = (found / known).rename("coverage").dropna().rename_axis(["series", "year"]).reset_index()
print("coverage = 4-digit rows / chapter total for the drilled-into chapters")
display(cov.pivot(index="series", columns="year", values="coverage").loc[:, BENCHMARK_YEARS].map(lambda v: f"{v:.1%}"))
latest_known = chap[(chap.year == LATEST) & chap.series.isin(drill_series)].set_index(["series", "cmdCode"])["value_usd_m"]
latest_found = hs4_data[hs4_data.year == LATEST].groupby(["series", "chapter"]).value_usd_m.sum().rename_axis(["series", "cmdCode"])
chapter_cov = pd.DataFrame({"chapter_total_usd_m": latest_known, "four_digit_usd_m": latest_found}).dropna(subset=["chapter_total_usd_m"]).fillna(0)
chapter_cov = chapter_cov[chapter_cov.index.map(lambda ix: ix[1] in drill_chapters[ix[0]])]
chapter_cov["coverage"] = chapter_cov["four_digit_usd_m"] / chapter_cov["chapter_total_usd_m"]
short_cov = chapter_cov[chapter_cov["coverage"] < 0.95].copy()
short_cov["chapter"] = [short_name(ix[1]) for ix in short_cov.index]
print(f"drilled-into chapters whose 4-digit rows cover under 95% of the chapter total in {LATEST} (product rankings omit the rest):")
display(short_cov if len(short_cov) else "none: every drilled-into chapter is fully covered")
print("chapters drilled into:", {k: [short_name(c) for c in v] for k, v in drill_chapters.items()})

In [ ]:
def top_products(series, n=10):
    p = hs4_data[hs4_data.series == series].pivot(index="cmdCode", columns="year", values="value_usd_m").reindex(columns=BENCHMARK_YEARS).fillna(0) / 1000
    out = p.loc[p[LATEST].nlargest(n).index].copy()
    out.insert(0, "product", [product_label(c) for c in out.index])
    return out


print("US exports to India: top products in the latest year (USD bn)")
display(top_products("us_exports"))
print("US imports from India: top products in the latest year (USD bn)")
display(top_products("us_imports"))

## 6. Visualizations

One value axis per chart, colour used for identity (a commodity chapter keeps its colour in every chart where it appears), thin marks, and charts also saved to `figures/`.

In [ ]:
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"
PALETTE = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED]
INK, INK2, MUTED, GRID, BASE, SURFACE, OTHER = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb", "#c3c2b7"
SEQ_BLUE = {1995: "#9ec5f4", 2005: "#5598e7", 2015: "#256abf", 2024: "#0d366b"}   # ordered years: light to dark

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "figure.dpi": 110,
    "axes.edgecolor": BASE, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False,
})


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


# one colour per commodity chapter, shared between the two directions wherever a chapter appears in both
TOP_K = 7
cum = {k: chap[chap.series == k].groupby("cmdCode").value_usd_m.sum() for k in ("us_exports", "us_imports")}
top_sets = {k: cum[k].nlargest(TOP_K).index.tolist() for k in cum}
shared = [c for c in top_sets["us_exports"] if c in top_sets["us_imports"]]
chapter_color = {c: PALETTE[i] for i, c in enumerate(shared)}
colors_for = {}
for k in cum:
    used = {chapter_color[c] for c in top_sets[k] if c in chapter_color}
    free = [p for p in PALETTE if p not in used]
    colors_for[k] = {c: chapter_color.get(c) or free.pop(0) for c in top_sets[k]}

### 6.1 Size of the trade and the balance

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9.5, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1.6]})
ax1.plot(tot.index, tot["us_exports"], color=BLUE, lw=2.2, label="US exports to India")
ax1.plot(tot.index, tot["us_imports"], color=ORANGE, lw=2.2, label="US imports from India")
for col, color in (("us_exports", BLUE), ("us_imports", ORANGE)):
    ax1.scatter([tot.index[-1]], [tot[col].iloc[-1]], s=40, color=color, edgecolor=SURFACE, linewidth=1.5, zorder=3)
    ax1.annotate(f"{tot[col].iloc[-1]:,.0f}", (tot.index[-1], tot[col].iloc[-1]), xytext=(6, -3), textcoords="offset points", fontsize=9, color=INK2)
ax1.set_ylabel("USD billion, current prices")
ax1.set_title("US goods trade with India (US customs data)")
ax1.legend(loc="upper left")
ax2.bar(tot.index, tot["us_balance"], width=0.7, color=[BLUE if v >= 0 else RED for v in tot["us_balance"]])
ax2.axhline(0, color=BASE, lw=1)
ax2.set_ylabel("USD billion")
ax2.set_title("US trade balance with India (blue = surplus, red = deficit)")
ax2.set_xlim(tot.index.min() - 1, tot.index.max() + 3)
fig.tight_layout()
save(fig, "01_totals_and_balance")
plt.show()

### 6.2 How the mix has changed

In [ ]:
def stacked_share(ax, series, title):
    p, share = share_table(series)
    order = top_sets[series]
    other = 100 - share.loc[order].sum()
    ys = [share.loc[c] for c in order] + [other]
    labels = [short_name(c) for c in order] + ["All other chapters"]
    cols = [colors_for[series][c] for c in order] + [OTHER]
    ax.stackplot(share.columns, ys, labels=labels, colors=cols, edgecolor=SURFACE, linewidth=0.8)
    ax.set_xlim(share.columns.min(), share.columns.max())
    ax.set_ylim(0, 100)
    ax.set_ylabel("share of the year's total (%)")
    ax.set_title(title)
    h, l = ax.get_legend_handles_labels()
    ax.legend(h[::-1], l[::-1], loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9)


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10.5, 9))
stacked_share(ax1, "us_exports", "What the US sells to India (share of US exports to India)")
stacked_share(ax2, "us_imports", "What the US buys from India (share of US imports from India)")
fig.tight_layout()
save(fig, "02_composition_over_time")
plt.show()

In [ ]:
def dumbbell_shares(ax, series, title, n=8):
    _, share = share_table(series)
    first_y = min(BENCHMARK_YEARS)
    chosen = sorted(set(share[LATEST].nlargest(n).index) | set(share[first_y].nlargest(n).index), key=lambda c: share.loc[c, LATEST])
    y = np.arange(len(chosen))
    ax.hlines(y, share.loc[chosen, first_y], share.loc[chosen, LATEST], color=BASE, lw=2, zorder=1)
    ax.scatter(share.loc[chosen, first_y], y, s=70, color=ORANGE, edgecolor=SURFACE, linewidth=1.5, zorder=2, label=str(first_y))
    ax.scatter(share.loc[chosen, LATEST], y, s=70, color=BLUE, edgecolor=SURFACE, linewidth=1.5, zorder=2, label=str(LATEST))
    ax.set_yticks(y, [short_name(c) for c in chosen])
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("share of the year's total (%)")
    ax.set_title(title)
    ax.legend(loc="lower right")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
dumbbell_shares(ax1, "us_exports", f"US exports to India: share, {min(BENCHMARK_YEARS)} vs {LATEST}")
dumbbell_shares(ax2, "us_imports", f"US imports from India: share, {min(BENCHMARK_YEARS)} vs {LATEST}")
fig.tight_layout()
save(fig, "03_share_then_vs_now")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for ax, series, title in ((ax1, "us_exports", "US exports to India"), (ax2, "us_imports", "US imports from India")):
    d = shifts(series, n=6).sort_values("change_pp")
    ax.barh([short_name(c) for c in d.index], d["change_pp"], height=0.62, color=[BLUE if v >= 0 else RED for v in d["change_pp"]])
    ax.axvline(0, color=BASE, lw=1)
    ax.grid(axis="y", visible=False)
    for i, v in enumerate(d["change_pp"]):
        ax.text(v + (0.3 if v >= 0 else -0.3), i, f"{v:+.1f}", va="center", ha="left" if v >= 0 else "right", fontsize=8.5, color=INK2)
    ax.margins(x=0.15)
    ax.set_xlabel("change in share of the total (percentage points)")
    ax.set_title(f"{title}: biggest shifts, {min(BENCHMARK_YEARS)} to {LATEST}")
fig.tight_layout()
save(fig, "04_gainers_and_losers")
plt.show()

### 6.3 Concentration

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(conc["us_exports"].index, conc["us_exports"]["top5_share_pct"], color=BLUE, lw=2.2, label="US exports to India")
ax.plot(conc["us_imports"].index, conc["us_imports"]["top5_share_pct"], color=ORANGE, lw=2.2, label="US imports from India")
ax.set_ylim(0, 100)
ax.set_ylabel("share of the year's total (%)")
ax.set_title("How concentrated is the trade? Share held by the top 5 chapters")
ax.legend(loc="lower right")
save(fig, "05_concentration")
plt.show()

### 6.4 Do the two countries' reports agree?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.4), gridspec_kw={"width_ratios": [1, 1.1]})
m = mirror.dropna(how="all")
ax1.plot(m.index, m["us_exports_vs_india_imports"], color=BLUE, lw=2.2, label="US exports: India's imports ÷ US exports")
ax1.plot(m.index, m["india_exports_vs_us_imports"], color=ORANGE, lw=2.2, label="India's exports: US imports ÷ India's exports")
ax1.axhline(1.0, color=BASE, lw=1, ls=(0, (4, 3)))
ax1.text(m.index.max(), 1.0, "perfect agreement ", ha="right", va="bottom", fontsize=8.5, color=MUTED)
ax1.set_ylabel("importer-reported ÷ exporter-reported")
ax1.set_title("Two sides of the same trade")
ax1.legend(loc="upper left", fontsize=9)

g = chapter_gap("in_exports", "us_imports", n=8).sort_values("importer_reported")
y = np.arange(len(g))
ax2.hlines(y, g["exporter_reported"] / 1000, g["importer_reported"] / 1000, color=BASE, lw=2, zorder=1)
ax2.scatter(g["exporter_reported"] / 1000, y, s=70, color=ORANGE, edgecolor=SURFACE, linewidth=1.5, zorder=2, label="India's reported exports")
ax2.scatter(g["importer_reported"] / 1000, y, s=70, color=BLUE, edgecolor=SURFACE, linewidth=1.5, zorder=2, label="US reported imports")
ax2.set_yticks(y, g["chapter"])
ax2.grid(axis="y", visible=False)
ax2.set_xlabel("USD billion")
ax2.set_title(f"India to US, {MIRROR_YEAR}: where the two reports differ most")
ax2.legend(loc="lower right")
fig.tight_layout()
save(fig, "06_mirror_gap")
plt.show()

### 6.5 What drove the growth?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.4))
for ax, series, title in ((ax1, "us_exports", "US exports to India"), (ax2, "us_imports", "US imports from India")):
    tbl, total_change = contributions(series, n=8)
    tbl = tbl.sort_values("change_usd_bn")
    ax.barh(tbl["chapter"], tbl["change_usd_bn"], height=0.62, color=[BLUE if v >= 0 else RED for v in tbl["change_usd_bn"]])
    ax.axvline(0, color=BASE, lw=1)
    ax.grid(axis="y", visible=False)
    for i, v in enumerate(tbl["change_usd_bn"]):
        ax.text(v + 0.2, i, f"{v:,.1f}", va="center", fontsize=8.5, color=INK2)
    ax.margins(x=0.15)
    ax.set_xlabel("change in USD billion")
    ax.set_title(f"{title}: largest contributors, {GROWTH_START} to {LATEST}\n(total change {total_change:,.1f} bn)")
fig.tight_layout()
save(fig, "07_growth_contributors")
plt.show()

### 6.6 Product-level detail

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
for ax, series, title in ((ax1, "us_exports", "US exports to India"), (ax2, "us_imports", "US imports from India")):
    t = top_products(series, n=10).iloc[::-1]
    y = np.arange(len(t))
    for yr in BENCHMARK_YEARS:
        ax.scatter(t[yr], y, s=60, color=SEQ_BLUE[yr], edgecolor=SURFACE, linewidth=1.2, zorder=3, label=str(yr))
    ax.set_yticks(y, [f"{p} ({c})" for c, p in zip(t.index, t["product"])], fontsize=9)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("USD billion")
    ax.set_title(f"{title}: top 10 products in {LATEST}")
    ax.legend(loc="lower right", title="year", ncol=2)
fig.tight_layout()
save(fig, "08_top_products")
plt.show()

## 7. Save the clean data

In [ ]:
chapter_out = chap.assign(chapter=chap["cmdCode"].map(chapter_names), short_name=chap["cmdCode"].map(short_name),
                          series_label=chap["series"].map(lambda k: SERIES[k]["label"]))
chapter_out.to_csv(CLEAN_DIR / "chapter_trade_1995_2024.csv", index=False)
hs4_data.to_csv(CLEAN_DIR / "product_trade_benchmark_years.csv", index=False)
tot.round(4).to_csv(CLEAN_DIR / "totals_by_year.csv")
print("saved:", ", ".join(p.name for p in sorted(CLEAN_DIR.glob("*.csv"))))

## 8. Insights and conclusions

All figures are from the data downloaded for this notebook (US-reported unless stated otherwise), in current US dollars. They describe **goods trade only**.

### What the data shows

1. **Trade has grown about 14-fold, and the US deficit has grown faster.** Two-way goods trade rose from 9.4 bn USD in 1995 to 133.0 bn in 2024 (about 9.6% a year in nominal terms). US exports to India grew from 3.3 bn to 41.8 bn (12.7x) and imports from India from 6.1 bn to 91.2 bn (15.0x). The US recorded a deficit with India in every one of the 30 years, widening from 2.8 bn to 49.5 bn (17.7x). Growth was not smooth: both flows dipped in 2009 and 2020, imports jumped 70% between 2020 and 2022 (53.6 bn to 91.0 bn), and US exports peaked at 47.3 bn in 2022 and have been around 40 to 42 bn since.

2. **What the US sells to India has been reshaped by energy.** In 1995 the top exports were machinery (24.7% of the total), electrical machinery (13.2%) and fertilizers (8.3%). By 2024 mineral fuels were 30.2% (12.6 bn USD, up from 2.5%), led by crude petroleum (6.5 bn, essentially zero in 2015), coal (3.2 bn) and petroleum gases (1.4 bn). Fuel exports to India were 1.1 bn in 2015 and 12.6 bn in 2024, and fuels alone account for 36% of the 33.8 bn increase in US exports since 2005. Machinery fell from 24.7% to 7.9% and fertilizers from 8.3% to almost nothing. Gems and precious metals (diamonds and gold) peaked at 30.7% of exports in 2015 and were 12.7% in 2024. Export trade has also become more concentrated: three chapters (fuels, gems, machinery) make up 51% of 2024 exports, and the top-5 share rose from 58% to 63%.

3. **What the US buys from India has moved from gems and clothing to pharmaceuticals and electronics.** In 1995 gems and jewelry (28.5%) and non-knitted apparel (16.3%) were 45% of US imports from India. In 2024 the top three chapters were electrical machinery and electronics (15.8%, 14.4 bn), pharmaceuticals (13.95%, 12.7 bn) and gems (13.0%, 11.9 bn), and apparel had fallen to 2.8%. Pharmaceuticals were 0.02% of imports in 1995; packaged medicaments alone were 12.2 bn in 2024, against 0.3 bn in 2005 and 5.9 bn in 2015. Telephone sets including smartphones went from 0.08 bn in 2015 to 9.0 bn in 2024. Electronics and pharmaceuticals together account for 36% of the 71.4 bn rise in US imports since 2005. Even so, imports became less concentrated overall (HHI from 1,196 in 1995 to 772 in 2024), because the old reliance on gems and apparel faded.

4. **The two countries' customs figures do not agree, and the gap matters for the headline.** By US figures the US ran a 49.5 bn deficit with India in 2024. India's own reports imply the opposite sign from India's side: it recorded 79.4 bn of exports to the US against 39.0 bn of imports from it, a 40.5 bn surplus, about 9 bn smaller than the US figure. On India's exports, the US records 15% more than India does (the median over the 30 years; the US figure is higher in 29 of 30 years). Some excess is expected, because importing countries value goods with freight and insurance included, but that is typically 5 to 10%. In 2024 the excess was concentrated in pharmaceuticals (US 12.7 bn vs India 8.7 bn), gems (11.9 vs 9.1 bn), electronics (14.4 vs 12.3 bn) and a 1.4 bn "unclassified" category that the US records and India barely does. On US exports the pattern is reversed in 2024: India records only 93% of what the US says it exported, mostly in gems (US 5.3 bn vs India 2.3 bn), aircraft (3.0 bn vs 0.8 bn) and unclassified goods, partly offset by machinery (India 5.8 bn vs US 3.3 bn). This ratio is volatile, between 0.78 (2000) and 1.31 (2008). I did not test why; timing, valuation, classification and third-country routing are all possible.

### Caveats

* **Goods only.** Services (software, IT, travel, education) are a large part of US–India economic ties and are not in this data, so this is a partial picture of the relationship.
* **Current dollars.** Nothing is adjusted for inflation, so the 14-fold growth overstates real growth, and the fuel and gold figures move with prices.
* **Which side to trust.** The main series are US customs figures. As point 4 shows, India's reports give different totals and, for several chapters, different sizes, so any single "deficit" number depends on whose records you use.
* **The aircraft gap in the product tables.** For US exports, the 4-digit rows in the drilled-into chapters add up to only 91% of the chapter totals in 2024 and 89% in 2005. The shortfall is entirely **Aircraft & parts**: only 0.14 bn of its 3.0 bn chapter total appears at 4-digit level in 2024. The main aircraft heading is not returned for this partner (I have not confirmed why; confidentiality suppression is one possibility), so the export product ranking leaves out about 2.9 bn, around 7% of 2024 exports, which would otherwise rank among the top products. All other drilled-into chapters, and all US imports, are fully covered.
* **Classification changes.** The Harmonized System was revised in 1996, 2002, 2007, 2012, 2017 and 2022. Chapter-level (2-digit) figures are fairly stable across revisions, but 4-digit product comparisons across decades are less so.
* **Revisions and the latest year.** Countries revise their trade figures after first publication, so 2024 values may change.
* **Source terms.** UN Comtrade restricts redistribution and commercial use of its data and requires attribution. For that reason the downloaded data and the cleaned CSVs are not committed to the repository; the notebook rebuilds them.